# World Model Car — Autonomous driving with a world model (Dreamer)
シミュレータ側（学習）

Future University Hakodate, 2022 Systems Information Science Practice, Group 22-A

This notebook is based on Dreamer_PyTorch by Kaito Suzuki (MIT License).
  https://github.com/cross32768/Dreamer_PyTorch
  Copyright (c) 2020 Kaito Suzuki

Modified for Donkey Car autonomous driving by the World Model Car team
(Jin Nakamura, Atsuya Katogi, Seiji Ito, Sotaro Kuroiwa, Yuto Watanabe), 2022.

本ノートブックは Kaito Suzuki 氏の Dreamer_PyTorch (MIT License) を土台に、
Donkey Car での自動運転向けに改変したものです。

In [ ]:
#!pip install pybullet

In [ ]:
import time
import os
import shutil

import csv

import gym

import numpy as np
import matplotlib.pyplot as plt
# import gym_donkeycar

import torch
import cv2

# from gym.wrappers import ResizeObservation
from torch.distributions import Normal
from torch.distributions.kl import kl_divergence
from torch import nn
from torch.nn import functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.tensorboard import SummaryWriter

# 以下のインポートはこちらのサイトから： https://gym-donkeycar.readthedocs.io/en/latest/_modules/gym_donkeycar/envs/donkey_env.html#DonkeyEnv.step
import random
from gym import spaces
import gym_donkeycar
from gym.utils import seeding
# from gym_donkeycar.envs.donkey_sim import DonkeyUnitySimContoller
from gym_donkeycar.envs.donkey_env import DonkeyEnv


print("ここで使用する環境を設定してください")
print("2つ以上インポートしている場合は、def make_envも変更するようにしてください。")
from gym_donkeycar.envs.donkey_env import GeneratedRoadsEnv
# from gym_donkeycar.envs.donkey_env import WarehouseEnv
# from gym_donkeycar.envs.donkey_env import AvcSparkfunEnv
# from gym_donkeycar.envs.donkey_env import GeneratedTrackEnv
# from gym_donkeycar.envs.donkey_env import MountainTrackEnv
# from gym_donkeycar.envs.donkey_env import GeneratedTrackEnv
# from gym_donkeycar.envs.donkey_env import RoboRacingLeagueTrackEnv
# from gym_donkeycar.envs.donkey_env import WaveshareEnv
from gym_donkeycar.envs.donkey_env import MiniMonacoEnv
# from gym_donkeycar.envs.donkey_env import WarrenTrackEnv
# from gym_donkeycar.envs.donkey_env import ThunderhillTrackEnv
# from gym_donkeycar.envs.donkey_env import CircuitLaunchEnv



from gym_donkeycar.envs.donkey_proc import DonkeyUnityProcess

#画像保存
from PIL import Image
import datetime
import pytz

%load_ext tensorboard


# actionの調和平均を取るため
# from statistics import harmonic_mean
import pandas as pd

In [ ]:
import torch
torch.__version__

In [ ]:
import sys
print(sys.version)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
gym.__version__

In [ ]:
torch.cuda.device_count()

In [ ]:
!nvidia-smi

In [ ]:
# env.reset()
# plt.imshow(image)
# plt.show()
# env.close()

# SET UP ENVIRONMENT
# You can also launch the simulator separately
# in that case, you don't need to pass a `conf` object


### 以下の内容は、nake_envの方で実行している
# exe_path = f"<PATH_TO_DONKEY_SIM_EXECUTABLE>"
# port = 9091
# conf = { "exe_path" : exe_path, "port" : port }
# env = gym.make("donkey-generated-roads-v0", conf=conf)
# env.reset()
# env.close()




In [ ]:
class GymWrapper_PyBullet(object):
    """
    """
    metadata = {'render.modes': ['human', 'rgb_array']}
    reward_range = (-np.inf, np.inf)

    def __init__(self, env, cam_dist=3, cam_yaw=0, cam_pitch=-30, render_width=120, render_height=160):
        self._env = env
#         self._env.env._cam_dist = cam_dist
#         self._env.env._cam_yaw = cam_yaw
#         self._env.env._cam_pitch = cam_pitch
        self._env.env._render_width = render_width
        self._env.env._render_height = render_height

    def __getattr(self, name):
        return getattr(self._env, name)

    @property
    def observation_space(self):
        width = self._env.env._render_width
        height = self._env.env._render_height
        return gym.spaces.Box(0, 255, (height, width, 3), dtype=np.uint8)

    @property
    def action_space(self):
        return self._env.action_space

    def step(self, action):
        _, reward, done, info = self._env.step(action)
        obs = self._env.render(mode="rgb_array")
#         obs = self._env.render()
        
        return obs, reward, done, info

    def reset(self):
        self._env.reset()
        obs = self._env.render(mode="rgb_array")
#         obs = self._env.render()
        
        return obs

    def render(self, mode='human', **kwargs):
        return self._env.render(mode, **kwargs)

    def close(self):
        self._env.close()

In [ ]:
# env = gym.make('HalfCheetahBulletEnv-v0')
# env = GymWrapper_PyBullet(env, cam_dist=2, cam_pitch=0, render_width=64, render_height=64)

In [ ]:
# env.reset()
# image = env.render(mode='rgb_array')
# plt.imshow(image)
# plt.show()
# env.close()

In [ ]:
class RepeatAction(gym.Wrapper):
    """
    """
    def __init__(self, env, skip=4):
        gym.Wrapper.__init__(self, env)
        self._skip = skip

    def reset(self):
        return self.env.reset()

    def step(self, action):
        total_reward = 0.0
        for _ in range(self._skip):
            obs, reward, done, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, info

In [ ]:
def make_env():
    exe_path = f"<PATH_TO_DONKEY_SIM_EXECUTABLE>"
    port = 9091
    conf = { "exe_path" : exe_path, "port" : port, "cam_resolution": [64,64,3], "max_cte": 5.0} #50.0 }
#     env = gym.make("donkey-generated-roads-v0", conf=conf)
#     env = gym.make(gym.Env)

#     env = GymWrapper_PyBullet(env, cam_dist=2, cam_pitch=0, render_width=64, render_height=64)
    #env = ResizeObservation(env, (3, 64, 64))
#     gym.spaces.Box(0, 255, (64, 64, 3), dtype=np.uint8)
#     env = np.resize(env, (3, 64, 64))
#     def __init__(self, level, time_step=0.05, frame_skip=2):
#     env = DonkeyEnv(level="generated_track", conf = conf)
#     env = GeneratedRoadsEnv(conf = conf)
#     env = GeneratedTrackEnv(conf = conf)
    env = MiniMonacoEnv(conf = conf)
    
    
#     env = DonkeyEnv(level=3, time_step=0.05, frame_ski)
#     env = GeneratedTrackEnv()
    return env

In [ ]:
class TransitionModel(nn.Module):
    """
    """
    def __init__(self, state_dim, action_dim, rnn_hidden_dim,
                 hidden_dim=200, min_stddev=0.1, act=F.elu):
        """
        action_dim: env.action_space.shape[0], 行動環境の次元?
        min_stddev: バイアス? ノイズ?
        """
        super(TransitionModel, self).__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.rnn_hidden_dim = rnn_hidden_dim
        
        self.fc_state_action = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc_rnn_hidden = nn.Linear(rnn_hidden_dim, hidden_dim)
        self.fc_state_mean_prior = nn.Linear(hidden_dim, state_dim)
        self.fc_state_stddev_prior = nn.Linear(hidden_dim, state_dim)
        self.fc_rnn_hidden_embedded_obs = nn.Linear(rnn_hidden_dim + 1024, hidden_dim)

        self.fc_state_mean_posterior = nn.Linear(hidden_dim, state_dim)
        self.fc_state_stddev_posterior = nn.Linear(hidden_dim, state_dim)

        self.rnn = nn.GRUCell(hidden_dim, rnn_hidden_dim)
        self._min_stddev = min_stddev
        self.act = act
  

    def forward(self, state, action, rnn_hidden, embedded_next_obs):
        """
        h_t+1 = f(h_t, s_t, a_t)

        """
        next_state_prior, rnn_hidden = self.prior(self.reccurent(state, action, rnn_hidden))

        # 疑問: なぜposteriorはreccurentを呼び出さないのか?→上のpriorからh_t+1(rnn_hidden)が求まっているから
        next_state_posterior = self.posterior(rnn_hidden, embedded_next_obs)

        return next_state_prior, next_state_posterior, rnn_hidden
      
    def reccurent(self, state, action, rnn_hidden):
        """
        """
        hidden = self.act(self.fc_state_action(torch.cat([state, action], dim=1)))

        # h_t+1 = f(hidden, h_t)
        rnn_hidden = self.rnn(hidden, rnn_hidden) 

        return rnn_hidden

    def prior(self, rnn_hidden):
        """
        """
        """
        """
        hidden = self.act(self.fc_rnn_hidden(rnn_hidden))

        # 全結合層に入力し, 平均を求める(?) 
        mean = self.fc_state_mean_prior(hidden)

        # 決定的状態を全結合層に通し, その結果を活性化関数(ソフトプラス)に通した値を標準偏差とする?
        stddev = F.softplus(self.fc_state_stddev_prior(hidden)) + self._min_stddev

        return Normal(mean, stddev), rnn_hidden

    def posterior(self, rnn_hidden, embedded_obs):
        """
        """
        # q(s_t+1 | h_t+1, e_t+1)の h_t, o_tがhiddenに対応?
        hidden = self.act(self.fc_rnn_hidden_embedded_obs(torch.cat([rnn_hidden, embedded_obs], dim=1)))
        
        mean = self.fc_state_mean_posterior(hidden)
        
        stddev = F.softplus(self.fc_state_stddev_posterior(hidden)) + self._min_stddev

        return Normal(mean, stddev)

In [ ]:
class ObservationModel(nn.Module):
    """
    p(o_t | s_t, h_t)
    """
    def __init__(self, state_dim, rnn_hidden_dim):
        super(ObservationModel, self).__init__()
        self.fc = nn.Linear(state_dim + rnn_hidden_dim, 1024)
        self.dc1 = nn.ConvTranspose2d(1024, 128, kernel_size=5, stride=2)
        self.dc2 = nn.ConvTranspose2d(128, 64, kernel_size=5, stride=2)
        self.dc3 = nn.ConvTranspose2d(64, 32, kernel_size=6 , stride=2)
        self.dc4 = nn.ConvTranspose2d(32, 3, kernel_size=6 , stride=2)
#         self.dc1 = nn.ConvTranspose2d(1024, 128, kernel_size=[11, 15], stride=2)
#         self.dc2 = nn.ConvTranspose2d(128, 64, kernel_size=[7, 8], stride=2)
#         self.dc3 = nn.ConvTranspose2d(64, 32, kernel_size=[6,7] , stride=2)
#         self.dc4 = nn.ConvTranspose2d(32, 3, kernel_size=[6,8] , stride=2)

    def forward(self, state, rnn_hidden):
        hidden = self.fc(torch.cat([state, rnn_hidden], dim=1))
        hidden = hidden.view(hidden.size(0), 1024, 1, 1)
        hidden = F.relu(self.dc1(hidden))
        hidden = F.relu(self.dc2(hidden))
        hidden = F.relu(self.dc3(hidden))
        obs = self.dc4(hidden)
        return obs

In [ ]:
class RewardModel(nn.Module):
    """
    p(r_t | s_t, h_t)
    """
    def __init__(self, state_dim, rnn_hidden_dim, hidden_dim=400, act=F.elu):
        super(RewardModel, self).__init__()
        self.fc1 = nn.Linear(state_dim + rnn_hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, 1)
        self.act = act
 

    def forward(self, state, rnn_hidden):
        hidden = self.act(self.fc1(torch.cat([state, rnn_hidden], dim=1)))
        hidden = self.act(self.fc2(hidden))
        hidden = self.act(self.fc3(hidden))
        reward = self.fc4(hidden)
        return reward

In [ ]:
class RSSM:
    def __init__(self, state_dim, action_dim, rnn_hidden_dim, ):
        self.transition = TransitionModel(state_dim, action_dim, rnn_hidden_dim).to(device)
        self.observation = ObservationModel(state_dim, rnn_hidden_dim,).to(device)
        self.reward = RewardModel(state_dim, rnn_hidden_dim,).to(device)

## 4.1 action[0]を移動平均にする

In [ ]:
# step_of_mean = 20 #7
# action0_data = np.zeros(step_of_mean)
num_for_action0_assignment = 0

In [ ]:
# step_of_mean = 20 #7

# def take_action_moving_average(num_for_action0_assignment, action):
# #     print(action0_data)
#     action0_data[num_for_action0_assignment % step_of_mean] = action[0]
#     action[0] = sum(action0_data)/len(action0_data)
# #     print(action0_data)
#     num_for_action0_assignment += 1
# #     print("num_for_action0_assignment", num_for_action0_assignment)
#     return num_for_action0_assignment, action

In [ ]:
# step_of_mean = 20 #7

# ### 加重調和平均模索
# def take_action_moving_average(num_for_action0_assignment, action):
# #     print(action0_data)
#     action0_data[num_for_action0_assignment % step_of_mean] = action[0]
#     weighte_sum = 0
#     for i in range(step_of_mean):
# #         if (num_for_action0_assignment-i)<0:
# #             weighte_sum += 0
# #         else:
#         weighte_sum += (step_of_mean-i) * action0_data[ (num_for_action0_assignment-i) % step_of_mean ]
# #         print("step_of_mean-i",step_of_mean-i)
# #         print("(num_for_action0_assignment-i) % step_of_mean",(num_for_action0_assignment-i) % step_of_mean)
# #         print("action0_data[ (num_for_action0_assignment-i) % step_of_mean ",action0_data[ (num_for_action0_assignment-i) % step_of_mean ])
#         print()
    
#     action[0] = weighte_sum/ (( step_of_mean**2)/2 )
# #     print("action0_data",action0_data)
# #     print("action[0]",action[0])
#     num_for_action0_assignment += 1
# #     print("num_for_action0_assignment", num_for_action0_assignment)
#     return num_for_action0_assignment, action

In [ ]:
step_of_mean = 20
action0_data = pd.Series(0)

def take_action_moving_average(action0_data, num_for_action0_assignment, action):
#     action0 = pd.Series(action[0])
# #     print(action0)
#     action0_data = pd.concat([action0_data[-step_of_mean:], action0], ignore_index=True)
# #     action0_data = pd.concat([action0_data, action0], ignore_index=True)

# #     print("action0_data",action0_data)
#     action0_mod = pd.DataFrame(action0_data).ewm(span=step_of_mean).mean().iloc[-1].tolist()[0]
#     action[0] = action0_mod
#     action0_data[-1:] = action0_mod
    
#     print("action[0]",action[0] )
    return action0_data, num_for_action0_assignment, action
    

In [ ]:
class ReplayBuffer:
    def __init__(self, memory_size):
        self.memory_size = memory_size
        self.memory = deque([], maxlen = memory_size)
    
    def append(self, transition):
        self.memory.append(transition)
    
    def sample(self, batch_size):
        batch_indexes = np.random.randint(0, len(self.memory), size=batch_size)
        states      = np.array([self.memory[index]['state'] for index in batch_indexes])
        next_states = np.array([self.memory[index]['next_state'] for index in batch_indexes])
        rewards     = np.array([self.memory[index]['reward'] for index in batch_indexes])
        actions     = np.array([self.memory[index]['action'] for index in batch_indexes])
        dones   = np.array([self.memory[index]['done'] for index in batch_indexes])
        return {'states': states, 'next_states': next_states, 'rewards': rewards, 'actions': actions, 'dones': dones}

In [ ]:
class ReplayBuffer(object):
    """
    """
    def __init__(self, capacity, observation_shape, action_dim):
        self.capacity = capacity

        self.observations = np.zeros((capacity, *observation_shape), dtype=np.uint8)
#         self.observations = np.zeros((capacity, 64, 64, 3), dtype = np.uint8)
        
        self.actions = np.zeros((capacity, action_dim), dtype=np.float32)
        self.rewards = np.zeros((capacity, 1), dtype=np.float32)
        self.done = np.zeros((capacity, 1), dtype=np.bool)

        self.index = 0
        self.is_filled = False

    def push(self, observation, action, reward, done):
        """
        """
#         print("push_obs", observation)
#         print("push_obs.shape", observation.shape)
        
#         print("--------------push--------------")
        self.observations[self.index] = observation
#         print("observations",self.observations)
#         print("observations.shape",self.observations.shape)
        
        self.actions[self.index] = action
        self.rewards[self.index] = reward
        self.done[self.index] = done

        if self.index == self.capacity - 1:
            self.is_filled = True
        self.index = (self.index + 1) % self.capacity

    def sample(self, batch_size, chunk_length):
        """
        """
        episode_borders = np.where(self.done)[0]
        sampled_indexes = []
        for _ in range(batch_size):
            cross_border = True
            while cross_border:
                initial_index = np.random.randint(len(self) - chunk_length + 1)
                final_index = initial_index + chunk_length - 1

                cross_border = np.logical_and(initial_index <= episode_borders,
                                              episode_borders < final_index).any()
            sampled_indexes += list(range(initial_index, final_index + 1))

        sampled_observations = self.observations[sampled_indexes].reshape(
            batch_size, chunk_length, *self.observations.shape[1:])
        sampled_actions = self.actions[sampled_indexes].reshape(
            batch_size, chunk_length, self.actions.shape[1])
        sampled_rewards = self.rewards[sampled_indexes].reshape(
            batch_size, chunk_length, 1)
        sampled_done = self.done[sampled_indexes].reshape(
            batch_size, chunk_length, 1)
        return sampled_observations, sampled_actions, sampled_rewards, sampled_done

    def __len__(self):
        return self.capacity if self.is_filled else self.index

In [ ]:
def preprocess_obs(obs):
    """
    """
    obs = obs.astype(np.float32)
    normalized_obs = obs / 255.0 - 0.5
    return normalized_obs

In [ ]:
def lambda_target(rewards, values, gamma, lambda_):
    """
    """
    V_lambda = torch.zeros_like(rewards, device=rewards.device)

    H = rewards.shape[0] - 1
    V_n = torch.zeros_like(rewards, device=rewards.device)
    V_n[H] = values[H]

    for n in range(1, H+1):
        V_n[:-n] = (gamma ** n) * values[n:]
        for k in range(1, n+1):
            if k == n:
                V_n[:-n] += (gamma ** (n-1)) * rewards[k:]
            else:
                V_n[:-n] += (gamma ** (k-1)) * rewards[k:-n+k]

        if n == H:
            V_lambda += (lambda_ ** (H-1)) * V_n
        else:
            V_lambda += (1 - lambda_) * (lambda_ ** (n-1)) * V_n # 最終的にV_lambdaは1に収束する?

    return V_lambda # 返された後に平均が求められる

In [ ]:
class Encoder(nn.Module):
    """
    """
    def __init__(self):
        super(Encoder, self).__init__()
        self.cv1 = nn.Conv2d(3, 32, kernel_size=4, stride=2)
        self.cv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.cv3 = nn.Conv2d(64, 128, kernel_size=4, stride=2)
        self.cv4 = nn.Conv2d(128, 256, kernel_size=4, stride=2)

    def forward(self, obs):
        hidden = F.relu(self.cv1(obs))
        hidden = F.relu(self.cv2(hidden))
        hidden = F.relu(self.cv3(hidden))
        embedded_obs = F.relu(self.cv4(hidden)).reshape(hidden.size(0), -1)
        return embedded_obs

In [ ]:
class ValueModel(nn.Module):
    """
    """
    def __init__(self, state_dim, rnn_hidden_dim, hidden_dim=400, act=F.elu):
        super(ValueModel, self).__init__()
        self.fc1 = nn.Linear(state_dim + rnn_hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, 1)
        self.act = act

    def forward(self, state, rnn_hidden):
        hidden = self.act(self.fc1(torch.cat([state, rnn_hidden], dim=1)))
        hidden = self.act(self.fc2(hidden))
        hidden = self.act(self.fc3(hidden))
        state_value = self.fc4(hidden)
        return state_value

In [ ]:
class ActionModel(nn.Module):
    """
    """
    def __init__(self, state_dim, rnn_hidden_dim, action_dim,
                 hidden_dim=400, act=F.elu, min_stddev=1e-4, init_stddev=5.0):
        super(ActionModel, self).__init__()
        self.fc1 = nn.Linear(state_dim + rnn_hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_mean = nn.Linear(hidden_dim, action_dim)
        self.fc_stddev = nn.Linear(hidden_dim, action_dim)
        self.act = act
        self.min_stddev = min_stddev
        self.init_stddev = np.log(np.exp(init_stddev) - 1)

    def forward(self, state, rnn_hidden, training=True):
        """
        """
        hidden = self.act(self.fc1(torch.cat([state, rnn_hidden], dim=1)))
        hidden = self.act(self.fc2(hidden))
        hidden = self.act(self.fc3(hidden))
        hidden = self.act(self.fc4(hidden))

        mean = self.fc_mean(hidden)
        mean = 5.0 * torch.tanh(mean / 5.0)
        stddev = self.fc_stddev(hidden)
        stddev = F.softplus(stddev + self.init_stddev) + self.min_stddev
        if training:
            action = torch.tanh(Normal(mean, stddev).rsample())
        else:
            action = torch.tanh(mean)
        return action

In [ ]:
class Agent:
    """
    """
    def __init__(self, encoder, rssm, action_model):
        self.encoder = encoder
        self.rssm = rssm
        self.action_model = action_model

        self.device = next(self.action_model.parameters()).device
        self.rnn_hidden = torch.zeros(1, rssm.rnn_hidden_dim, device=self.device)

    def __call__(self, obs, training=True):
        obs = preprocess_obs(obs)
        obs = torch.as_tensor(obs, device=self.device)
        obs = obs.transpose(1, 2).transpose(0, 1).unsqueeze(0)

        with torch.no_grad():
            embedded_obs = self.encoder(obs)
            state_posterior = self.rssm.posterior(self.rnn_hidden, embedded_obs)
            state = state_posterior.sample()
            action = self.action_model(state, self.rnn_hidden, training=training)

            _, self.rnn_hidden = self.rssm.prior(self.rssm.reccurent(state, action, self.rnn_hidden))

        return action.squeeze().cpu().numpy()

    def reset(self):
        self.rnn_hidden = torch.zeros(1, self.rssm.rnn_hidden_dim, device=self.device)

In [ ]:
env = make_env()

# buffer_capacity = 200000
buffer_capacity = 300000  # Colabのメモリの都合上, 元の実装より小さめにとっています
replay_buffer = ReplayBuffer(capacity=buffer_capacity,
                              observation_shape=env.observation_space.shape,
                              action_dim=env.action_space.shape[0])
print("observation_shape",env.observation_space.shape)
state_dim = 30
rnn_hidden_dim = 200
encoder = Encoder().to(device)
rssm = RSSM(state_dim,env.action_space.shape[0],rnn_hidden_dim, )
print("rssm done")
value_model = ValueModel(state_dim, rnn_hidden_dim).to(device)
action_model = ActionModel(state_dim, rnn_hidden_dim,
                             env.action_space.shape[0]).to(device)

# PATH = "model_save/2022-11-25 18:17:42.109092+09:00599.pth"
# checkpoint = torch.load(PATH)
# encoder.load_state_dict(checkpoint['encoder_state_dict'])
# rssm.transition.load_state_dict(checkpoint['rssm_state_dict'])
# rssm.observation.load_state_dict(checkpoint['observation_state_dict'])
# rssm.reward.load_state_dict(checkpoint['reward_state_dict'])
# value_model.load_state_dict(checkpoint['value_state_dict'])
# action_model.load_state_dict(checkpoint['action_state_dict'])





model_lr = 6e-4
value_lr = 8e-5
action_lr = 8e-5
eps = 1e-4
model_params = (list(encoder.parameters()) +
                  list(rssm.transition.parameters()) +
                  list(rssm.observation.parameters()) +
                  list(rssm.reward.parameters()))
# print("model_params done")
model_optimizer = torch.optim.Adam(model_params, lr=model_lr, eps=eps)
# print("model_optimizer done")
value_optimizer = torch.optim.Adam(value_model.parameters(), lr=value_lr, eps=eps)
# print("value_optimizer done")
action_optimizer = torch.optim.Adam(action_model.parameters(), lr=action_lr, eps=eps)
# print("action_optimizer done")

# model_optimizer.load_state_dict(checkpoint['model_optimizer'])
# value_optimizer.load_state_dict(checkpoint['value_optimizer'])
# action_optimizer.load_state_dict(checkpoint['action_optimizer'])

test_interval = 10

#テスト用
seed_episodes = 5 # 最初にランダム行動で探索するエピソード数
all_episodes = 600  # 学習全体のエピソード数（300ほどで, ある程度収束します）
# test_interval = 7  # 何エピソードごとに探索ノイズなしのテストを行うか
model_save_interval =100  # NNの重みを何エピソードごとに保存するか
collect_interval = 100  # 何回のNNの更新ごとに経験を集めるか（＝1エピソード経験を集めるごとに何回更新するか）

# for state in optimizer.state.values():
#     for k, v in state.items():
#         if isinstance(v, torch.Tensor):
#             state[k] = v.to(device)
# epoch = checkpoint['epoch']
# model_loss = checkpoint['model_loss']
# kl_loss = checkpoint['kl_loss']
# obs_loss = checkpoint['obs_loss']
# reward_loss = checkpoint['reward_loss']
# value_loss = checkpoint['value_loss:']
# action_loss = checkpoint['action_loss']


# collect_interval = 100
action_noise_var = 0.3

batch_size = 50
chunk_length = 50
imagination_horizon = 15


gamma = 0.9
lambda_ = 0.95
clip_grad_norm = 100
free_nats = 1e-7 #3  # KL誤差（RSSMのTransitionModelにおけるpriorとposteriorの間の誤差）がこの値以下の場合, 無視する

# env.close()

In [ ]:
# import torch.onnx 
# import onnx

# #Function to Convert to ONNX 
# def Convert_ONNX(): 

#     # set the model to inference mode 
# #     model.eval()
#     model.train(False)

#     # Let's create a dummy input tensor  
# #     dummy_input = torch.randn(1, input_size, requires_grad=True)  

#     # Export the model   
#     torch.onnx.export(model,         # model being run 
#          dummy_input,       # model input (or a tuple for multiple inputs) 
#          "ImageClassifier.onnx",       # where to save the model  
#          export_params=True,  # store the trained parameter weights inside the model file 
#          opset_version=10,    # the ONNX version to export the model to 
#          do_constant_folding=True,  # whether to execute constant folding for optimization 
#          input_names = ['modelInput'],   # the model's input names 
#          output_names = ['modelOutput'], # the model's output names 
#          dynamic_axes={'modelInput' : {0 : 'batch_size'},    # variable length axes 
#                                 'modelOutput' : {0 : 'batch_size'}}) 
#     print(" ") 
#     print('Model has been converted to ONNX')


# if __name__ == "__main__": 

#     # Let's build our model 
#     #train(5) 
#     #print('Finished Training') 

#     # Test which classes performed well 
#     #testAccuracy() 

#     # Let's load the model we just created and test the accuracy per label 

#     model = ActionModel(state_dim, rnn_hidden_dim, env.action_space.shape[0])
#     dummy_input = (torch.randn(1, state_dim), torch.randn(1, rnn_hidden_dim))
#     path = "<PATH_TO_MODEL_CHECKPOINT>" 
# #     model.load_state_dict(torch.load(path)) 
#     checkpoint = torch.load(path)
#     model.load_state_dict(checkpoint['action_state_dict'])
#     # Test with batch of images 
#     #testBatch() 
#     # Test how the classes performed 
#     #testClassess() 
 
#     # Conversion to ONNX 
#     Convert_ONNX()

In [ ]:
# import torch
# import numpy as np
# # Save the weights of a PyTorch model to a file
# torch.save(model.state_dict(), "model_weights.pt")
# # Load the weights into a NumPy array
# weights = torch.load("model_weights.pt")
# weights_array = np.array(weights)
# # Load the weights into a new PyTorch model
# new_model = MyModel()
# new_model.load_state_dict(torch.from_numpy(weights_array)

In [ ]:
# # translate_weights.py

# from collections import OrderedDict
# import numpy as np
# import shutil
# import torch
# import sys
# import os

# # PyTorchの異なるバージョン間での重みの変換
# # 例 PyTorch 1.8 で学習 -> PyTorch 1.1 で推論

# name_action = "<PATH_TO_MODEL_DIRECTORY>/action_model.pth"
# name_encoder = "<PATH_TO_MODEL_DIRECTORY>/encoder.pth"
# name_obs = "<PATH_TO_MODEL_DIRECTORY>/obs_model.pth"
# name_reward = "<PATH_TO_MODEL_DIRECTORY>/reward_model.pth"
# name_rssm = "<PATH_TO_MODEL_DIRECTORY>/rssm.pth"
# name_value = "<PATH_TO_MODEL_DIRECTORY>/value_model.pth"
# topdire_name="tmp12_5/"
# # os.mkdir(topdire_name)
# model_name = name_action
# tmp_name="tmp_action"
# if not os.path.isdir(tmp_name):
#   # 変換元のPyTorchで実行 (例 PyTorch 1.8)

#     os.mkdir(tmp_name)

#     ckpt = torch.load(model_name)
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#         np.save(f"tmp_action/{k}.npy", v)
#     print("done")

# model_name = name_encoder
# tmp_name="tmp_encoder"
# if not os.path.isdir(tmp_name):
#   # 変換元のPyTorchで実行 (例 PyTorch 1.8)

#     os.mkdir(tmp_name)

#     ckpt = torch.load(model_name)
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#         np.save(f"tmp_encoder/{k}.npy", v)
#     print("done")

# model_name = name_obs
# tmp_name="tmp_obs"
# if not os.path.isdir(tmp_name):
#   # 変換元のPyTorchで実行 (例 PyTorch 1.8)

#     os.mkdir(tmp_name)

#     ckpt = torch.load(model_name)
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#         np.save(f"tmp_obs/{k}.npy", v)
#     print("done")

# model_name = name_reward
# tmp_name="tmp_reward"
# if not os.path.isdir(tmp_name):
#   # 変換元のPyTorchで実行 (例 PyTorch 1.8)

#     os.mkdir(tmp_name)

#     ckpt = torch.load(model_name)
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#         np.save(f"tmp_reward/{k}.npy", v)
#     print("done")
    
# model_name = name_rssm
# tmp_name="tmp_rssm"
# if not os.path.isdir(tmp_name):
#   # 変換元のPyTorchで実行 (例 PyTorch 1.8)

#     os.mkdir(tmp_name)

#     ckpt = torch.load(model_name)
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#         np.save(f"tmp_rssm/{k}.npy", v)
#     print("done")

# model_name = name_value
# tmp_name="tmp_value"
# if not os.path.isdir(tmp_name):
#   # 変換元のPyTorchで実行 (例 PyTorch 1.8)

#     os.mkdir(tmp_name)

#     ckpt = torch.load(model_name)
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#     for k, v in ckpt.items():
#         print("k, v", k, v)
#         v = v.to("cpu").detach().numpy()
#         k = k.replace(".", "+")
#         np.save(f"tmp_value/{k}.npy", v)
#     print("done")
    
    
    
    
    
# else:
#   # 変換後のPyTorchで実行 (例 PyTorch 1.1)

#     ckpt = OrderedDict()

#     for k in os.listdir("tmp/"):
#         v = np.load(f"tmp/{k}")
#         k = k[:-4].replace("+", ".")

#         ckpt[k] = torch.from_numpy(v)

#     torch.save(ckpt, model_name)

#     shutil.rmtree("tmp")

In [ ]:
# print(onnx.__version__)

In [ ]:
start_time = datetime.datetime.now(pytz.timezone('Asia/Tokyo'))
print("開始時刻：", start_time)

In [ ]:
now_time = datetime.datetime.now(pytz.timezone('Asia/Tokyo'))
dir = 'data' + '/'
os.makedirs(dir, exist_ok = True)

for episode in range(seed_episodes):
    action0_data = pd.Series(0)
    obs = env.reset()
    obs = cv2.resize(obs,dsize = (64,64))
    
    dir = 'data/sim_img_data'+str(now_time) + "/"
    os.makedirs(dir, exist_ok = True)
    dir = dir + 'episode'+ str(episode) + "/"
    os.makedirs(dir, exist_ok = True)
    
    done = False
    c=0
    while not done:
        plt.imshow(obs)
        plt.show()
        obs = cv2.resize(obs,dsize = (64,64))
        
        pil_img = Image.fromarray(obs)
        if  c%15 == 0:

            filename=dir + str(c) + '.png'
            pil_img.save(filename)
            
        action = env.action_space.sample()
        action0_data, num_for_action0_assignment, action = take_action_moving_average(action0_data, num_for_action0_assignment, action)
        
        action[1]=(action[1]+0.75)/2
        next_obs, reward, done, _ = env.step(action)
        replay_buffer.push(obs, action, reward, done)
        obs = next_obs
        print(c)
        c+=1
        if c ==2000:
            done = True
            
        


In [ ]:
action0_data.plot()

In [ ]:
log_dir = 'logs_test_from1031'
#log_dir = 'logs'
writer = SummaryWriter(log_dir)

In [ ]:
# for episode in range(seed_episodes, all_episodes):
#     if (episode + 1) % model_save_interval == 0:
#         now=format(datetime.datetime.now(pytz.timezone('Asia/Tokyo')),'%Y%m%d%H%M%S')
#         model_log_dir = os.path.join(log_dir,'episode_%04d' % (episode + 1)+"_"+now)
#         print(now,'episode_%04d' % (episode + 1)+"_"+now)

In [ ]:
# for episode in range(epoch, all_episodes):

for episode in range(seed_episodes, all_episodes):
    action0_data = pd.Series(0)
    # -----------------------------
    # -----------------------------
    start = time.time()
    policy = Agent(encoder, rssm.transition, action_model)
    obs = env.reset()
    
    dir = 'data/sim_img_data'+str(now_time) + "/"
    os.makedirs(dir, exist_ok = True)
    dir = dir + 'episode'+ str(episode) + "/"
    os.makedirs(dir, exist_ok = True)
    c=0
    
    done = False
    total_reward = 0
    while not done:
#         print("learning c",c)
        obs = cv2.resize(obs,dsize = (64,64))
        
        pil_img = Image.fromarray(obs)
        if  c%15 == 0:
            filename=dir + str(c) + '.png'
            pil_img.save(filename)
            print("filename", filename)
        
        action = policy(obs)
        
        # ステアリングの移動平均
#         print(action)
        action0_data, num_for_action0_assignment, action = take_action_moving_average(action0_data, num_for_action0_assignment, action)
        # スロットルの調節
        action[1]=(action[1]+0.75)/2
#         print(action)
#         print(action)
        action += np.random.normal(0, np.sqrt(action_noise_var),
                                     env.action_space.shape[0])        
        
        next_obs, reward, done, _ = env.step(action)
        replay_buffer.push(obs, action, reward, done)
        
        obs = next_obs
        total_reward += reward
        
        c += 1
        if c ==2000:
            done = True
        
    writer.add_scalar('total reward at train', total_reward, episode)
    print('episode [%4d/%4d] is collected. Total reward is %f' %
            (episode+1, all_episodes, total_reward))
    print('elasped time for interaction: %.2fs' % (time.time() - start))

    start = time.time()
    for update_step in range(collect_interval):
        # -------------------------------------------------------------------------------------
        # -------------------------------------------------------------------------------------
        observations, actions, rewards, _ = \
            replay_buffer.sample(batch_size, chunk_length)

        observations = preprocess_obs(observations)
        observations = torch.as_tensor(observations, device=device)
        observations = observations.transpose(3, 4).transpose(2, 3)
        observations = observations.transpose(0, 1)
        actions = torch.as_tensor(actions, device=device).transpose(0, 1)
        rewards = torch.as_tensor(rewards, device=device).transpose(0, 1)

        print("update_step",update_step)
        embedded_observations = encoder(
#             observations.reshape(-1, 3, 120, 160)).view(chunk_length, batch_size, -1)
            observations.reshape(-1, 3, 64, 64)).view(chunk_length, batch_size, -1)

        states = torch.zeros(chunk_length, batch_size, state_dim, device=device)
        rnn_hiddens = torch.zeros(chunk_length, batch_size, rnn_hidden_dim, device=device)

        state = torch.zeros(batch_size, state_dim, device=device)
        rnn_hidden = torch.zeros(batch_size, rnn_hidden_dim, device=device)

        kl_loss = 0
        for l in range(chunk_length-1):
            next_state_prior, next_state_posterior, rnn_hidden = \
                rssm.transition(state, actions[l], rnn_hidden, embedded_observations[l+1])
            state = next_state_posterior.rsample()
            states[l+1] = state
            rnn_hiddens[l+1] = rnn_hidden
            kl = kl_divergence(next_state_prior, next_state_posterior).sum(dim=1)
            kl_loss += kl.clamp(min=free_nats).mean()
            
            
        kl_loss /= (chunk_length - 1)

        states = states[1:]
        rnn_hiddens = rnn_hiddens[1:]

        flatten_states = states.view(-1, state_dim)
        flatten_rnn_hiddens = rnn_hiddens.view(-1, rnn_hidden_dim)
        #recon_observations = rssm.observation(flatten_states, flatten_rnn_hiddens).view(chunk_length-1, batch_size, 3, 120, 160)
        recon_observations = rssm.observation(flatten_states, flatten_rnn_hiddens).view(chunk_length-1, batch_size, 3, 64, 64)
        predicted_rewards = rssm.reward(flatten_states, flatten_rnn_hiddens).view(chunk_length-1, batch_size, 1)

        obs_loss = 0.5 * F.mse_loss(recon_observations, observations[1:], reduction='none').mean([0, 1]).sum()
        reward_loss = 0.5 * F.mse_loss(predicted_rewards, rewards[:-1])

        model_loss = kl_loss + obs_loss + reward_loss
        model_optimizer.zero_grad()
        model_loss.backward()
        clip_grad_norm_(model_params, clip_grad_norm)
        model_optimizer.step()

        print("-----------------------")
        print(kl_loss.grad_fn)
        print("-----------------------")

        # --------------------------------------------------
        # --------------------------------------------------
        flatten_states = flatten_states.detach()
        flatten_rnn_hiddens = flatten_rnn_hiddens.detach()

        imaginated_states = torch.zeros(imagination_horizon + 1,
                                         *flatten_states.shape,
                                          device=flatten_states.device)
        imaginated_rnn_hiddens = torch.zeros(imagination_horizon + 1,
                                                *flatten_rnn_hiddens.shape,
                                                device=flatten_rnn_hiddens.device)

        imaginated_states[0] = flatten_states
        imaginated_rnn_hiddens[0] = flatten_rnn_hiddens
        
        for h in range(1, imagination_horizon + 1):
            actions = action_model(flatten_states, flatten_rnn_hiddens)
            flatten_states_prior, flatten_rnn_hiddens = rssm.transition.prior(rssm.transition.reccurent(flatten_states,
                                                                   actions,
                                                                   flatten_rnn_hiddens))
            flatten_states = flatten_states_prior.rsample()
            imaginated_states[h] = flatten_states
            imaginated_rnn_hiddens[h] = flatten_rnn_hiddens

        flatten_imaginated_states = imaginated_states.view(-1, state_dim)
        flatten_imaginated_rnn_hiddens = imaginated_rnn_hiddens.view(-1, rnn_hidden_dim)
        imaginated_rewards = \
            rssm.reward(flatten_imaginated_states,
                        flatten_imaginated_rnn_hiddens).view(imagination_horizon + 1, -1)
        imaginated_values = \
            value_model(flatten_imaginated_states,
                        flatten_imaginated_rnn_hiddens).view(imagination_horizon + 1, -1)

        lambda_target_values = lambda_target(imaginated_rewards, imaginated_values, gamma, lambda_)

        action_loss = -lambda_target_values.mean()
        action_optimizer.zero_grad()
        action_loss.backward()
        clip_grad_norm_(action_model.parameters(), clip_grad_norm)
        action_optimizer.step()
        
        imaginated_values = value_model(flatten_imaginated_states.detach(), flatten_imaginated_rnn_hiddens.detach()).view(imagination_horizon + 1, -1)        
        value_loss =  0.5 * F.mse_loss(imaginated_values, lambda_target_values.detach())
        value_optimizer.zero_grad()
        value_loss.backward()
        clip_grad_norm_(value_model.parameters(), clip_grad_norm)
        value_optimizer.step()

#         #ログをcsvに出力
        
#         with open('csv_data/model_writer_row.csv', 'a') as f:
#             writer = csv.writer(f)
#             writer.writerows(model_loss)
#         with open('csv_data/kl_writer_row.csv', 'a') as f:
#             writer = csv.writer(f)
#             writer.writerows(kl_loss)
#         with open('csv_data/obs_writer_row.csv', 'a') as f:
#             writer = csv.writer(f)
#             writer.writerows(obs_loss)
#         with open('csv_data/reward_writer_row.csv', 'a') as f:
#             writer = csv.writer(f)
#             writer.writerows(reward_loss)
#         with open('csv_data/value_writer_row.csv', 'a') as f:
#             writer = csv.writer(f)
#             writer.writerows(value_loss)
#         with open('csv_data/action_writer_row.csv', 'a') as f:
#             writer = csv.writer(f)
#             writer.writerows(action_loss)
        
        print('update_step: %3d model loss: %.5f, kl_loss: %.5f, '
             'obs_loss: %.5f, reward_loss: %.5f, '
             'value_loss: %.5f action_loss: %.5f'
                % (update_step + 1, model_loss.item(), kl_loss.item(),
                    obs_loss.item(), reward_loss.item(),
                    value_loss.item(), action_loss.item()))
        total_update_step = episode * collect_interval + update_step
        writer.add_scalar('model loss', model_loss.item(), total_update_step)
        writer.add_scalar('kl loss', kl_loss.item(), total_update_step)
        writer.add_scalar('obs loss', obs_loss.item(), total_update_step)
        writer.add_scalar('reward loss', reward_loss.item(), total_update_step)
        writer.add_scalar('value loss', value_loss.item(), total_update_step)
        writer.add_scalar('action loss', action_loss.item(), total_update_step)

    print('elasped time for update: %.2fs' % (time.time() - start))

    # --------------------------------------------------------------
    # --------------------------------------------------------------
    if (episode + 1) % test_interval == 0:
        policy = Agent(encoder, rssm.transition, action_model)
        start = time.time()
        obs = env.reset()
        done = False
        total_reward = 0
        while not done:
            obs = cv2.resize(obs,dsize = (64,64))
            action = policy(obs, training=False)
            # ステアリングの移動平均
            action0_data, num_for_action0_assignment, action = take_action_moving_average(action0_data, num_for_action0_assignment, action)
            # スロットルの調節
            action[1]=(action[1]+0.75)/2
            obs, reward, done, _ = env.step(action)
            total_reward += reward

        writer.add_scalar('total reward at test', total_reward, episode)
        print('Total test reward at episode [%4d/%4d] is %f' %
                (episode+1, all_episodes, total_reward))
        print('elasped time for test: %.2fs' % (time.time() - start))

    if (episode + 1) % model_save_interval == 0:
        model_log_dir_name = str(now_time)+'/episode_%04d' % (episode + 1)+"_"
        print(model_log_dir_name)
        model_log_dir = os.path.join(log_dir, model_log_dir_name)
        if(os.path.isfile(model_log_dir) == True):
            shutil.rmtree(model_log_dir)
#         os.makedirs(model_log_dir)
#         os.rmtree(model_log_dir)
#         os.makedirs(model_log_dir)
        os.makedirs(model_log_dir, exist_ok=True)
        torch.save(encoder.state_dict(), os.path.join(model_log_dir, 'encoder.pth'))
        torch.save(rssm.transition.state_dict(), os.path.join(model_log_dir, 'rssm.pth'))
        torch.save(rssm.observation.state_dict(), os.path.join(model_log_dir, 'obs_model.pth'))
        torch.save(rssm.reward.state_dict(), os.path.join(model_log_dir, 'reward_model.pth'))
        torch.save(value_model.state_dict(), os.path.join(model_log_dir, 'value_model.pth'))
        torch.save(action_model.state_dict(), os.path.join(model_log_dir, 'action_model.pth'))
        
        ##モデルの保存（追加学習用）
#         os.makedirs('model_save/now_time', exist_ok = True)
        save_path = "model_save/" + str(now_time) + str(episode) + ".pth"
        torch.save({'epoch': episode,
            'encoder_state_dict': encoder.state_dict(),
            'rssm_state_dict': rssm.transition.state_dict(),
            'observation_state_dict': rssm.observation.state_dict(),
            'reward_state_dict': rssm.reward.state_dict(),
            'value_state_dict': value_model.state_dict(),
            'action_state_dict': action_model.state_dict(),
            'model_optimizer': model_optimizer.state_dict(),
            'action_optimizer': action_optimizer.state_dict(),
            'value_optimizer': value_optimizer.state_dict(),
            'model_loss': model_loss.item(),
            'kl_loss': kl_loss.item(),
            'obs_loss': obs_loss.item(),
            'reward_loss': reward_loss.item(),
            'value_loss:': value_loss.item(),
            'action_loss': action_loss.item(),
            },
           save_path)

writer.close()

In [ ]:
%tensorboard --logdir='./logs_test1006'

In [ ]:
print("開始時刻：", start_time)
end_time = datetime.datetime.now(pytz.timezone('Asia/Tokyo'))
print("開始時刻：", end_time)

In [ ]:

# from google_drive_downloader import GoogleDriveDownloader as gdd

# gdd.download_file_from_google_drive(file_id=file_id,
#                                        dest_path='./episode_0100.zip',
#                                        unzip=True)

In [ ]:
# encoder.load_state_dict(torch.load('./episode_0100/encoder.pth'))
# rssm.transition.load_state_dict(torch.load('./episode_0100/rssm.pth'))
# rssm.observation.load_state_dict(torch.load('./episode_0100/obs_model.pth'))
# action_model.load_state_dict(torch.load('./episode_0100/action_model.pth'))

In [ ]:
print("model_log_dir",model_log_dir)
print("save_path",save_path)
print(model_log_dir, 'encoder.pth')

In [ ]:
log_dir = 'logs_test_from1031'
model_log_dir_name = input("ディレクトリ名を「episode_00...」と指定してください。\n：続きでよろしければenterを押してください：") if input("ディレクトリ名を「episode_...」と指定してください：") !="" else model_log_dir_name
print("model_log_dir_name",model_log_dir_name)
model_log_dir = os.path.join(log_dir, model_log_dir_name)
# model_log_dir = os.path.join(log_dir, 'episode_0400')
encoder.load_state_dict(torch.load(os.path.join(model_log_dir, 'encoder.pth')))
rssm.transition.load_state_dict(torch.load(os.path.join(model_log_dir, 'rssm.pth')))
rssm.observation.load_state_dict(torch.load(os.path.join(model_log_dir, 'obs_model.pth')))
action_model.load_state_dict(torch.load(os.path.join(model_log_dir, 'action_model.pth')))

# rssm.reward.load_state_dict(torch.load(os.path.join(model_log_dir, 'reward_model.pth')))
# value_model.load_state_dict(torch.load(os.path.join(model_log_dir, 'value_model.pth')))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML


def display_video(frames):
    plt.figure(figsize=(8, 8), dpi=50)
    patch = plt.imshow(frames[0])
    plt.axis('off')
    
    def animate(i):
        patch.set_data(frames[i])
        plt.title("Step %d" % (i))
    
    anim = animation.FuncAnimation(plt.gcf(), animate, frames=len(frames), interval=50)
    display(HTML(anim.to_jshtml(default_mode='once')))
    plt.close()

In [ ]:
policy = Agent(encoder, rssm.transition, action_model)
env = make_env()
obs = env.reset() #一瞬コメントアウト外したので不具合あれば戻す10/06
done = False
total_reward = 0
frames = [obs]
for _ in range(0, 1):
    obs = env.reset()
    done = False
    while not done:
        obs = cv2.resize(obs, (64, 64))
        action = policy(obs, training=False)
        # ステアリングの移動平均
        action0_data, num_for_action0_assignment, action = take_action_moving_average(action0_data, num_for_action0_assignment, action)
        # スロットルの調節
        action[1]=(action[1]+0.75)/4
        next_obs, reward, done, _ = env.step(action)
        obs = next_obs
        total_reward += reward
        frames.append(obs)

# for i in range(0, 20):
#     obs = env.reset()
#     done = False
#     while not done:
#         obs = cv2.resize(obs, (64, 64))
#         action = policy(obs, training=False)
#         next_obs, reward, done, _ = env.step(action)
#         obs = next_obs
#         env.render()

print('Total Reward:', total_reward)

In [ ]:
display_video(frames)

In [ ]:
env = make_env()
reward_modelpolicy = Agent(encoder, rssm.transition, action_model)
obs = env.reset()
for _ in range(np.random.randint(5, 100)):
    obs = cv2.resize(obs, (64, 64))
    action = policy(obs, training=False)
    # ステアリングの移動平均
    action0_data, num_for_action0_assignment, action = take_action_moving_average(action0_data, num_for_action0_assignment, action)
    # スロットルの調節
    action[1]=(action[1]+0.75)/2
    obs, _, _, _ = env.step(action)
    obs = cv2.resize(obs, (64, 64))

preprocessed_obs = preprocess_obs(obs)
preprocessed_obs = torch.as_tensor(preprocessed_obs, device=device)
preprocessed_obs = preprocessed_obs.transpose(1, 2).transpose(0, 1).unsqueeze(0)
with torch.no_grad():
    embedded_obs = encoder(preprocessed_obs)

rnn_hidden = policy.rnn_hidden
state = rssm.transition.posterior(rnn_hidden, embedded_obs).sample()
frame = np.zeros((64, 128, 3))
frames = []

prediction_length = 100

obs = env.reset()
for _ in range(prediction_length):
    obs = cv2.resize(obs, (64, 64))
    action = policy(obs)
    # ステアリングの移動平均
    action0_data, num_for_action0_assignment, action = take_action_moving_average(action0_data, num_for_action0_assignment, action)
    # スロットルの調節
    action[1]=(action[1]+0.75)/2
    obs, _, _, _ = env.step(action)
    obs = cv2.resize(obs, (64, 64))

    action = torch.as_tensor(action, device=device).unsqueeze(0)
    action0_data, num_for_action0_assignment, action = take_action_moving_average(action0_data, num_for_action0_assignment, action)
    with torch.no_grad():
        state_prior, rnn_hidden = rssm.transition.prior(rssm.transition.reccurent(state, action, rnn_hidden))
        state = state_prior.sample()
        predicted_obs = rssm.observation(state, rnn_hidden)#obs_model(state, rnn_hidden)

    frame[:, :64, :] = preprocess_obs(obs)
    frame[:, 64:, :] = predicted_obs.squeeze().transpose(0, 1).transpose(1, 2).cpu().numpy()
    frames.append((frame + 0.5).clip(0.0, 1.0))

In [ ]:
display_video(frames)